# Modello 2: Feature Engineering e Ottimizzazione Quasi-Newton (L-BFGS)

Questo modello mira a validare l'ipotesi secondo cui una rappresentazione a bassa dimensionalità (11 feature euristiche) unita a un'ottimizzazione di ordine superiore può superare l'approccio Deep Learning "forza bruta" a parità di costo computazionale.

## L'Architettura (Regressione Lineare Pura)
Il modello è una singola trasformazione lineare, che ricalca esattamente la classica funzione di valutazione scacchistica (come quella usata da Shannon):
$$E = w_1 f_1 + w_2 f_2 + \dots + w_{11} f_{11} + b$$
Dove $f_i$ sono le nostre feature e $w_i$ sono i pesi da ottimizzare.

## L'Algoritmo: L-BFGS (Quasi-Newton)
Invece di usare la discesa del gradiente semplice $\mathbf{w}_{new} = \mathbf{w}_{old} - \alpha \nabla L$, usiamo un metodo Quasi-Newton che sfrutta la curvatura della funzione di costo:
$$\mathbf{w}_{new} = \mathbf{w}_{old} - \alpha B \nabla L(\mathbf{w}_{old})$$
La matrice $B$ è un'approssimazione iterativa dell'inversa della **Matrice Hessiana** ($H^{-1}$). Poiché abbiamo solo 12 parametri totali, $B$ è di dimensioni $12 \times 12$, rendendo il calcolo istantaneo e permettendo all'algoritmo di convergere al minimo in pochissimi step (epoche), senza i classici "zigzag" del gradiente puro.

In [5]:
# ==========================================
# CELLA 2: CARICAMENTO DATI (Notebook 3)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

print("Caricamento dati in corso...")
dati = np.load('dataset_elaborato.npz')

X_numpy = dati['X_13'] # AGGIORNATO PER CARICARE LE 13 COLONNE
Y_numpy = dati['Y'].copy()

Y_numpy = Y_numpy / 100.0
Y_numpy = np.tanh(Y_numpy / 4.0)

X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
Y_tensor = torch.tensor(Y_numpy, dtype=torch.float32).view(-1, 1)

print(f"✅ Dati caricati! Forma di X: {X_tensor.shape}")

Caricamento dati in corso...
✅ Dati caricati! Forma di X: torch.Size([336903, 13])


In [6]:
# ==========================================
# CELLA 3: MODELLO SHALLOW E TRAINING LOOP (Notebook 3)
# ==========================================
class ModelloEuristico(nn.Module):
    def __init__(self):
        super(ModelloEuristico, self).__init__()
        self.lineare = nn.Linear(13, 1) # AGGIORNATO A 13 INPUT

    def forward(self, x):
        return self.lineare(x)

modello_2 = ModelloEuristico()
funzione_costo = nn.MSELoss()
ottimizzatore = optim.LBFGS(modello_2.parameters(), lr=0.1, max_iter=20, history_size=10)

print("🚀 Inizio addestramento L-BFGS (13 feature)...")
epoche = 15 

for epoca in range(epoche):
    def closure():
        ottimizzatore.zero_grad()
        previsioni = modello_2(X_tensor) 
        loss = funzione_costo(previsioni, Y_tensor)
        loss.backward()
        return loss
    ottimizzatore.step(closure)
    print(f"✅ Epoca {epoca+1:02d}/{epoche} - Loss (MSE): {closure().item():.4f}")

torch.save(modello_2.state_dict(), 'pesi_modello_2_quasi_newton.pth')

pesi = modello_2.lineare.weight.data.numpy()[0]
bias = modello_2.lineare.bias.data.item()

nomi_feature = ["Materiale", "Coppia Alfieri", "Sbilanc. Qualità", "Mobilità", 
                "Centro", "Sviluppo", "Pedoni Passati", "Pedoni Doppiati", 
                "Pedoni Isolati", "Sicurezza Re", "Armonia", 
                "Torri Comunicanti", "Possibili Scacchi"] # LISTA AGGIORNATA

print("\n🧠 I PESI DEL MODELLO:")
for nome, peso in zip(nomi_feature, pesi): print(f"   {nome}: {peso:+.4f}")
print(f"   Bias: {bias:+.4f}")

🚀 Inizio addestramento L-BFGS (13 feature)...
✅ Epoca 01/15 - Loss (MSE): 0.4982
✅ Epoca 02/15 - Loss (MSE): 0.3780
✅ Epoca 03/15 - Loss (MSE): 0.3697
✅ Epoca 04/15 - Loss (MSE): 0.3670
✅ Epoca 05/15 - Loss (MSE): 0.3665
✅ Epoca 06/15 - Loss (MSE): 0.3664
✅ Epoca 07/15 - Loss (MSE): 0.3663
✅ Epoca 08/15 - Loss (MSE): 0.3663
✅ Epoca 09/15 - Loss (MSE): 0.3663
✅ Epoca 10/15 - Loss (MSE): 0.3662
✅ Epoca 11/15 - Loss (MSE): 0.3662
✅ Epoca 12/15 - Loss (MSE): 0.3662
✅ Epoca 13/15 - Loss (MSE): 0.3662
✅ Epoca 14/15 - Loss (MSE): 0.3662
✅ Epoca 15/15 - Loss (MSE): 0.3662

🧠 I PESI DEL MODELLO:
   Materiale: -0.0002
   Coppia Alfieri: -0.0047
   Sbilanc. Qualità: -0.0028
   Mobilità: +0.0002
   Centro: -0.0003
   Sviluppo: -0.0029
   Pedoni Passati: +0.1263
   Pedoni Doppiati: +0.0048
   Pedoni Isolati: -0.0927
   Sicurezza Re: +0.0019
   Armonia: +0.0008
   Torri Comunicanti: -0.0015
   Possibili Scacchi: -0.0021
   Bias: +0.0457
